# Model Training + MLflow Tracking
This notebook trains models and logs experiments using MLflow.


## MLflow Experiment Tracking


In [1]:
import os
from pathlib import Path
from datetime import datetime
import joblib

import pandas as pd
import mlflow
import mlflow.sklearn
import matplotlib.pyplot as plt

from mlflow.tracking import MlflowClient

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    roc_auc_score,
    ConfusionMatrixDisplay,
    RocCurveDisplay
)

# Load .env
from dotenv import load_dotenv
#load_dotenv()

# =====================================================
# Trial / Experiment Name
# =====================================================

trial_number = "_" + datetime.now().strftime("%Y%m%d_%H%M%S")

print("Experiment started at:", trial_number)

# =====================================================
# MLflow Tracking URI
# Priority:
# 1. DagsHub URI from ENV
# 2. Local MLflow fallback
# =====================================================

dagshub_uri = os.getenv("MLFLOW_TRACKING_URI")

if dagshub_uri:
    mlflow.set_tracking_uri(dagshub_uri)
    print(f"Using DagsHub MLflow URI: {dagshub_uri}")
else:
    tracking_path = Path("../mlruns").resolve()
    mlflow.set_tracking_uri(f"file:///{tracking_path.as_posix()}")
    print(f"Using Local MLflow URI: {tracking_path}")

Experiment started at: _20260517_203917
Using Local MLflow URI: C:\Bhooshaan_Local\BITS Local\Sem2\Assignments\DMML\DMML-Assignment\mlruns


In [2]:
runner = os.getenv("RUNNER_NAME", "local_runner")
print(f"Runner Name: {runner}")

Runner Name: local_runner


## Load Dataset & Train/Test Split

In [3]:
# =====================================================
# Load Dataset
# =====================================================

df = pd.read_csv('../data/processed/heart_clean.csv')

print(df.head())

# =====================================================
# Train Test Split
# =====================================================

X = df.drop('target', axis=1)
y = df['target']

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

print(f"Train size: {X_train.shape}, Test size: {X_test.shape}")

   age  sex  cp  trestbps  chol  fbs  restecg  thalach  exang  oldpeak  slope  \
0   63    1   1       145   233    1        2      150      0      2.3      3   
1   67    1   4       160   286    0        2      108      1      1.5      2   
2   67    1   4       120   229    0        2      129      1      2.6      2   
3   37    1   3       130   250    0        0      187      0      3.5      3   
4   41    0   2       130   204    0        2      172      0      1.4      1   

    ca  thal  target  
0  0.0   6.0       0  
1  3.0   3.0       1  
2  2.0   7.0       1  
3  0.0   3.0       0  
4  0.0   3.0       0  
Train size: (242, 13), Test size: (61, 13)


## Preprocessing

In [4]:
# =====================================================
# Preprocessing
# =====================================================

categorical_cols = ['cp', 'restecg', 'slope', 'thal']

numeric_cols = [
    col for col in X.columns
    if col not in categorical_cols
]

preprocessor = ColumnTransformer([
    ("num", StandardScaler(), numeric_cols),
    ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_cols)
])

print(f"Numeric columns  : {numeric_cols}")
print(f"Categorical cols : {categorical_cols}")

Numeric columns  : ['age', 'sex', 'trestbps', 'chol', 'fbs', 'thalach', 'exang', 'oldpeak', 'ca']
Categorical cols : ['cp', 'restecg', 'slope', 'thal']


## Helper Functions

In [5]:
# =====================================================
# Evaluation Function
# =====================================================

def evaluate(model):

    preds = model.predict(X_test)

    probs = model.predict_proba(X_test)[:, 1]

    return {
        "accuracy": accuracy_score(y_test, preds),
        "precision": precision_score(y_test, preds),
        "recall": recall_score(y_test, preds),
        "roc_auc": roc_auc_score(y_test, probs)
    }

# =====================================================
# Run Name Helper
# =====================================================

def format_run_name(model_name, params, runner=None):

    param_str = "_".join(
        [f"{k.split('__')[-1]}={v}" for k, v in params.items()]
    )

    if runner:
        return f"{model_name}_{param_str}_{runner}"
    return f"{model_name}_{param_str}"

# =====================================================
# Plot Logging Helper
# =====================================================

def log_evaluation_plots(model, run_name):

    artifacts_dir = Path("../artifacts")
    artifacts_dir.mkdir(parents=True, exist_ok=True)

    # Confusion matrix
    cm_path = artifacts_dir / f"{run_name}_confusion_matrix.png"
    fig_cm, ax_cm = plt.subplots(figsize=(6, 5))
    ConfusionMatrixDisplay.from_predictions(y_test, model.predict(X_test), ax=ax_cm)
    ax_cm.set_title(f"Confusion Matrix - {run_name}")
    fig_cm.tight_layout()
    fig_cm.savefig(cm_path, dpi=150)
    plt.close(fig_cm)

    # ROC curve
    roc_path = artifacts_dir / f"{run_name}_roc_curve.png"
    fig_roc, ax_roc = plt.subplots(figsize=(6, 5))
    RocCurveDisplay.from_estimator(model, X_test, y_test, ax=ax_roc)
    ax_roc.set_title(f"ROC Curve - {run_name}")
    fig_roc.tight_layout()
    fig_roc.savefig(roc_path, dpi=150)
    plt.close(fig_roc)

    mlflow.log_artifact(str(cm_path), artifact_path="plots")
    mlflow.log_artifact(str(roc_path), artifact_path="plots")

print("Helper functions defined.")

Helper functions defined.


## Model & Hyperparameter Configurations

In [6]:
# =====================================================
# Models and Hyperparameter Configurations
# =====================================================

models_and_params = {

    "logistic_regression": {
        "model": LogisticRegression(max_iter=1000),
        "params": [
            {"model__C": 0.1},
            {"model__C": 1},
            {"model__C": 10}
        ]
    },

    "random_forest": {
        "model": RandomForestClassifier(random_state=42),
        "params": [
            {
                "model__n_estimators": 100,
                "model__max_depth": 5
            },
            {
                "model__n_estimators": 100,
                "model__max_depth": 10
            },
            {
                "model__n_estimators": 200,
                "model__max_depth": 5
            },
            {
                "model__n_estimators": 200,
                "model__max_depth": 10
            }
        ]
    }
}

print("Model configurations ready.")

Model configurations ready.


## Experiment Tracking — Training Loop

In [7]:
# =====================================================
# Experiment Tracking
# =====================================================

experiment_name = "heart_disease_experiment_runs" + str(trial_number)

mlflow.set_experiment(experiment_name)

best_models = {}

for model_name, config in models_and_params.items():

    model = config["model"]
    param_list = config["params"]

    best_accuracy = -1
    best_run_id = None
    best_pipeline = None

    for params in param_list:

        pipeline = Pipeline([
            ("preprocessor", preprocessor),
            ("model", model)
        ])

        pipeline.set_params(**params)

        run_name = format_run_name(model_name, params, runner=runner)

        with mlflow.start_run(run_name=run_name):

            # ---------------------------------------------
            # Tags
            # ---------------------------------------------

            mlflow.set_tag("model_type", model_name)
            mlflow.set_tag("runner_name", run_name)

            # ---------------------------------------------
            # Train
            # ---------------------------------------------

            pipeline.fit(X_train, y_train)

            # ---------------------------------------------
            # Metrics
            # ---------------------------------------------

            metrics = evaluate(pipeline)

            # ---------------------------------------------
            # Log Params
            # ---------------------------------------------

            mlflow.log_params(params)

            # ---------------------------------------------
            # Log Metrics
            # ---------------------------------------------

            mlflow.log_metric("test_accuracy", metrics["accuracy"])
            mlflow.log_metric("test_precision", metrics["precision"])
            mlflow.log_metric("test_recall", metrics["recall"])
            mlflow.log_metric("test_roc_auc", metrics["roc_auc"])

            # ---------------------------------------------
            # Log Model
            # ---------------------------------------------

            mlflow.sklearn.log_model(
                pipeline,
                artifact_path="model"
            )

            # ---------------------------------------------
            # Log Evaluation Plots as Artifacts
            # ---------------------------------------------

            log_evaluation_plots(pipeline, run_name)

            print("=" * 60)
            print(f"Run Name : {run_name}")
            print(f"Accuracy : {metrics['accuracy']:.4f}")
            print(f"ROC AUC  : {metrics['roc_auc']:.4f}")
            print("=" * 60)

            # ---------------------------------------------
            # Track Best Model for Each Algorithm
            # ---------------------------------------------

            if metrics["accuracy"] > best_accuracy:

                best_accuracy = metrics["accuracy"]
                best_run_id = mlflow.active_run().info.run_id
                best_pipeline = pipeline

    best_models[model_name] = {
        "accuracy": best_accuracy,
        "run_id": best_run_id,
        "pipeline": best_pipeline
    }

2026/05/17 20:39:34 INFO mlflow.tracking.fluent: Experiment with name 'heart_disease_experiment_runs_20260517_203917' does not exist. Creating a new experiment.


Run Name : logistic_regression_C=0.1_local_runner
Accuracy : 0.8525
ROC AUC  : 0.9246


c:\Bhooshaan_Local\BITS Local\Sem2\Assignments\DMML\DMML-Assignment\dmmlenv\Lib\site-packages\_distutils_hack\__init__.py:18: UserWarning: Distutils was imported before Setuptools, but importing Setuptools also replaces the `distutils` module in `sys.modules`. This may lead to undesirable behaviors or errors. To avoid these issues, avoid using distutils directly, ensure that setuptools is installed in the traditional way (e.g. not an editable install), and/or make sure that setuptools is always imported before distutils.
  warnings.warn(
c:\Bhooshaan_Local\BITS Local\Sem2\Assignments\DMML\DMML-Assignment\dmmlenv\Lib\site-packages\_distutils_hack\__init__.py:33: UserWarning: Setuptools is replacing distutils.
  warnings.warn("Setuptools is replacing distutils.")


Run Name : logistic_regression_C=1_local_runner
Accuracy : 0.8361
ROC AUC  : 0.9073


c:\Bhooshaan_Local\BITS Local\Sem2\Assignments\DMML\DMML-Assignment\dmmlenv\Lib\site-packages\_distutils_hack\__init__.py:18: UserWarning: Distutils was imported before Setuptools, but importing Setuptools also replaces the `distutils` module in `sys.modules`. This may lead to undesirable behaviors or errors. To avoid these issues, avoid using distutils directly, ensure that setuptools is installed in the traditional way (e.g. not an editable install), and/or make sure that setuptools is always imported before distutils.
  warnings.warn(
c:\Bhooshaan_Local\BITS Local\Sem2\Assignments\DMML\DMML-Assignment\dmmlenv\Lib\site-packages\_distutils_hack\__init__.py:33: UserWarning: Setuptools is replacing distutils.
  warnings.warn("Setuptools is replacing distutils.")


...

Run Name : random_forest_n_estimators=100_max_depth=10_local_runner
Accuracy : 0.8525
ROC AUC  : 0.9370


c:\Bhooshaan_Local\BITS Local\Sem2\Assignments\DMML\DMML-Assignment\dmmlenv\Lib\site-packages\_distutils_hack\__init__.py:18: UserWarning: Distutils was imported before Setuptools, but importing Setuptools also replaces the `distutils` module in `sys.modules`. This may lead to undesirable behaviors or errors. To avoid these issues, avoid using distutils directly, ensure that setuptools is installed in the traditional way (e.g. not an editable install), and/or make sure that setuptools is always imported before distutils.
  warnings.warn(
c:\Bhooshaan_Local\BITS Local\Sem2\Assignments\DMML\DMML-Assignment\dmmlenv\Lib\site-packages\_distutils_hack\__init__.py:33: UserWarning: Setuptools is replacing distutils.
  warnings.warn("Setuptools is replacing distutils.")


Run Name : random_forest_n_estimators=200_max_depth=5_local_runner
Accuracy : 0.8852
ROC AUC  : 0.9440


c:\Bhooshaan_Local\BITS Local\Sem2\Assignments\DMML\DMML-Assignment\dmmlenv\Lib\site-packages\_distutils_hack\__init__.py:18: UserWarning: Distutils was imported before Setuptools, but importing Setuptools also replaces the `distutils` module in `sys.modules`. This may lead to undesirable behaviors or errors. To avoid these issues, avoid using distutils directly, ensure that setuptools is installed in the traditional way (e.g. not an editable install), and/or make sure that setuptools is always imported before distutils.
  warnings.warn(
c:\Bhooshaan_Local\BITS Local\Sem2\Assignments\DMML\DMML-Assignment\dmmlenv\Lib\site-packages\_distutils_hack\__init__.py:33: UserWarning: Setuptools is replacing distutils.
  warnings.warn("Setuptools is replacing distutils.")


Run Name : random_forest_n_estimators=200_max_depth=10_local_runner
Accuracy : 0.8689
ROC AUC  : 0.9429


## Register Best Models & Save Locally

In [8]:
# Register final best model to a consistent name for loading in next notebook

final_model_name = "heart_disease_best_model"

best_overall = max(best_models.items(), key=lambda x: x[1]["accuracy"])
best_overall_name = best_overall[0]
best_overall_info = best_overall[1]

model_uri = f"runs:/{best_overall_info['run_id']}/model"

mv = mlflow.register_model(
    model_uri=model_uri,
    name=final_model_name
)

print(f"Registered Final Model Name : {mv.name}")
print(f"Version : {mv.version}")

print(f"\nBest Overall Model: {best_overall_name} with Accuracy: {best_overall_info['accuracy']:.4f}")

Registered Final Model Name : heart_disease_best_model
Version : 1

Best Overall Model: random_forest with Accuracy: 0.8852


Successfully registered model 'heart_disease_best_model'.
Created version '1' of model 'heart_disease_best_model'.


In [9]:
# ---------------------------------------------
# Save Local PKL
# ---------------------------------------------

models_dir = Path("../models")

models_dir.mkdir(parents=True, exist_ok=True)

pkl_path = models_dir / f"{final_model_name}.pkl"

joblib.dump(best_overall_info["pipeline"], pkl_path)

print(f"Saved model locally to: {pkl_path.resolve()}")

Saved model locally to: C:\Bhooshaan_Local\BITS Local\Sem2\Assignments\DMML\DMML-Assignment\models\heart_disease_best_model.pkl



## Run MLflow UI

```bash
mlflow ui
```

Open:

http://127.0.0.1:5000



## DagsHub Integration

Set environment variable before running:

```bash
export MLFLOW_TRACKING_URI="https://dagshub.com/<username>/<repo>.mlflow"
```

Windows PowerShell:

```powershell
$env:MLFLOW_TRACKING_URI="https://dagshub.com/<username>/<repo>.mlflow"
```
